# Vertex Forecast: Time Series Dense Encoder (TiDE)

> Optimzed dense DNN-based encoder-decoder model. Good for fast training and inference which help with long contexts and horizons.

This notebook shows how to use the Vertex AI python SDK to run Vertex Forecast jobs

**references**
* Google AI Blog: [Recent advances in deep long-horizon forecasting](https://ai.googleblog.com/2023/04/recent-advances-in-deep-long-horizon.html)
* Research Paper: [Long-term Forecasting with TiDE: Time-series Dense Encoder](https://arxiv.org/abs/2304.08424)
* Documentation: [Best practices for creating tabular training data](https://cloud.google.com/vertex-ai/docs/tabular-data/bp-tabular)

In [23]:
import sys
sys.path.append("..")
import env_config

PROJECT_ID = env_config.PROJECT_ID
LOCATION = env_config.LOCATION
PREFIX = env_config.PREFIX

print(f"PREFIX: {PREFIX}")
print(f"PROJECT_ID: {PROJECT_ID}")
print(f"LOCATION: {LOCATION}")

PREFIX: vertex-forecast-v1
PROJECT_ID: hybrid-vertex
LOCATION: us-central1


### imports

In [46]:
import logging
logging.disable(logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

from google.cloud import aiplatform
from google.cloud import bigquery

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# clients
bq = bigquery.Client(project=PROJECT_ID)
aiplatform.init(project=PROJECT_ID, location=LOCATION)

### setup

In [3]:
REGION = LOCATION
EXPERIMENT = f'tide-python-{env_config.VERSION}'
SERIES = 'applied-forecasting'

BQ_PROJECT = PROJECT_ID
BQ_DATASET = f"a_vf_{env_config.VERSION}".replace('-','_')
BQ_TABLE = "forecast_1_prepped"

print(f"EXPERIMENT: {EXPERIMENT}")
print(f"SERIES: {SERIES}")

EXPERIMENT: tide-python-v1
SERIES: applied-forecasting


In [4]:
TARGET_COLUMN = 'num_trips'
TIME_COLUMN = 'starttime'
SERIES_COLUMN = 'start_station_name'
SPLIT_COLUMN = 'splits'
#COVARIATE_COLUMNS = ['avg_tripduration', 'pct_subscriber', 'ratio_gender', 'capacity'] # could be empty
COVARIATE_COLUMNS_ATTRIBUTES = []
COVARIATE_COLUMNS_KNOWN = ['capacity']
COVARIATE_COLUMNS_UNKNOWN = ['avg_tripduration', 'pct_subscriber', 'ratio_gender']

In [5]:
# CUSTOMIZE
FORECAST_GRANULARITY = 'DAY' # the data preparation included preparing the data at this level
FORECAST_HORIZON_LENGTH = 14
FORECAST_TEST_LENGTH = 14 # the data preparation included setting this value for splits = TEST
FORECAST_VALIDATE_LENGTH = 14 # the data preparation included setting this value for splits = VALIDATE

In [7]:
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"TIMESTAMP: {TIMESTAMP}")

TIMESTAMP: 20250331-224503


Retrieve Key Dates for splits:

In [11]:
query = f"""
    WITH
        SPLIT AS (
            SELECT splits, min({TIME_COLUMN}) as mindate, max({TIME_COLUMN}) as maxdate
            FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`
            GROUP BY {SPLIT_COLUMN}
        ),
        TRAIN AS (
            SELECT mindate as start_date
            FROM SPLIT
            WHERE {SPLIT_COLUMN} ='TRAIN'
        ),
        VAL AS (
            SELECT mindate as val_start
            FROM SPLIT
            WHERE {SPLIT_COLUMN} = 'VALIDATE'
        ),
        TEST AS (
            SELECT mindate as test_start, maxdate as end_date
            FROM SPLIT
            WHERE {SPLIT_COLUMN} = 'TEST'
        )
    SELECT * EXCEPT(pos) FROM
    (SELECT *, ROW_NUMBER() OVER() pos FROM TRAIN)
    JOIN (SELECT *, ROW_NUMBER() OVER() pos FROM VAL)
    USING (pos)
    JOIN (SELECT *, ROW_NUMBER() OVER() pos FROM TEST)
    USING (pos)
"""
# print(query)

keyDates = bq.query(query).to_dataframe()
keyDates

,start_date,val_start,test_start,end_date
0,2013-07-01,2016-09-03,2016-09-17,2016-09-30


Retrieve raw data:

In [44]:
query = f"""
    SELECT {SERIES_COLUMN}, {TIME_COLUMN}, {SPLIT_COLUMN}, {TARGET_COLUMN}
    FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`
    ORDER by {SERIES_COLUMN}, {TIME_COLUMN}
"""
# print(query)
rawSeries = bq.query(query).to_dataframe()

> TODO

# Create Vertex Forecast model 

## Create dataset

> create managed time series dataset with Vertex AI
* [Documentation](https://cloud.google.com/vertex-ai/docs/tabular-data/forecasting/create-dataset) for forecasting dataset
* [Python SDK reference](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.TimeSeriesDataset#google_cloud_aiplatform_TimeSeriesDataset_create) for `aiplatform.TimeSeriesDataset.create()`

In [14]:
if SERIES in [ds.display_name for ds in aiplatform.TimeSeriesDataset.list()]:
    dataset = aiplatform.TimeSeriesDataset.list(filter = f'display_name={SERIES}')[0]
else:
    dataset = aiplatform.TimeSeriesDataset.create(
        display_name = f'{SERIES}', 
        bq_source = f'bq://{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}',
        labels = {'series' : f'{SERIES}', 'experiment' : f'{EXPERIMENT}'}
    )

print(f'Created/Retrieve Dataset: {dataset.display_name}')

Created/Retrieve Dataset: applied-forecasting


In [15]:
dataset.column_names

['ratio_gender',
 'starttime',
 'num_trips',
 'splits',
 'avg_tripduration',
 'pct_subscriber',
 'start_station_name',
 'capacity']

## Train Forecasting Model with Vertex AI AutoML

### Search Model Registry

The result of an Vertex AI Forecasting Job is a model in the Vertex AI [Model Registry](https://cloud.google.com/vertex-ai/docs/model-registry/introduction). These models have versioning so if this notebook has been run before we could add a new run as a new verison to the same model in the registry. To do this we first check the model registry for an existing model from this notebook (using the SERIES and EXPERIMENT parameters).

In [16]:
modelmatch = aiplatform.Model.list(filter = f'display_name={SERIES}_{EXPERIMENT} AND labels.series={SERIES} AND labels.experiment={EXPERIMENT}')

if modelmatch:
    print("There is an existing model with versions: ", [f'{m.version_id}' for m in modelmatch])
    parent = modelmatch[0].resource_name
else:
    print("This is the first training for this model")
    parent = ''

This is the first training for this model


### Create TiDE forecast job

* [Documentation](https://cloud.google.com/vertex-ai/docs/tabular-data/forecasting/train-model#train_a_model) for forecasting job
* [Python SDK reference](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.TimeSeriesDenseEncoderForecastingTrainingJob) for `aiplatform.TimeSeriesDenseEncoderForecastingTrainingJob()`

In [17]:
column_specs = dict.fromkeys(
    list(
        set(dataset.column_names) - set([SPLIT_COLUMN, SERIES_COLUMN])
    ),
    'auto'
)
column_specs

{'ratio_gender': 'auto',
 'starttime': 'auto',
 'num_trips': 'auto',
 'avg_tripduration': 'auto',
 'pct_subscriber': 'auto',
 'capacity': 'auto'}

In [25]:
JOB_DISPLAY_NAME = f'{SERIES}_{EXPERIMENT}_{TIMESTAMP}'.replace('-','_')
JOB_DISPLAY_NAME

'applied_forecasting_tide_python_v1_20250331_224503'

In [26]:
forecasting_job = aiplatform.TimeSeriesDenseEncoderForecastingTrainingJob(
    display_name = JOB_DISPLAY_NAME,
    optimization_objective = "minimize-rmse",
    column_specs = column_specs,
    labels = {'series' : f'{SERIES}', 'experiment' : f'{EXPERIMENT}'}
)

forecasting_job

### Run TiDE Forecast job

Running the job is the point where the parameterization of the forecasting takes place.

`aiplatform.TimeSeriesDenseEncoderForecastingTrainingJob.run()`
* [Python SDK reference](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.TimeSeriesDenseEncoderForecastingTrainingJob#google_cloud_aiplatform_TimeSeriesDenseEncoderForecastingTrainingJob_run)
* [githib src](https://github.com/googleapis/python-aiplatform/blob/main/google/cloud/aiplatform/training_jobs.py#L1886)

In [27]:
PROJECT_ID

'hybrid-vertex'

In [28]:
forecast = forecasting_job.run(
    # data parameters
    dataset = dataset,
    target_column = TARGET_COLUMN,
    time_column = TIME_COLUMN,
    time_series_identifier_column = SERIES_COLUMN,
    time_series_attribute_columns = COVARIATE_COLUMNS_ATTRIBUTES,
    unavailable_at_forecast_columns = [TARGET_COLUMN] + COVARIATE_COLUMNS_UNKNOWN,
    available_at_forecast_columns = [TIME_COLUMN] + COVARIATE_COLUMNS_KNOWN,
    predefined_split_column_name = SPLIT_COLUMN,
    
    # forecast parameters
    forecast_horizon = FORECAST_HORIZON_LENGTH,
    data_granularity_unit = FORECAST_GRANULARITY,
    data_granularity_count = 1,
    context_window = 28,
    holiday_regions = ['GLOBAL', 'NA', 'US'],
    
    hierarchy_group_columns = [],
    hierarchy_group_total_weight = 1.0,
    hierarchy_temporal_total_weight = 2.0,
    hierarchy_group_temporal_total_weight = 1.0,
    
    # output parameters
    export_evaluated_data_items = True,
    export_evaluated_data_items_bigquery_destination_uri = f"bq://{BQ_PROJECT}:{BQ_DATASET}:{EXPERIMENT}_eval",
    export_evaluated_data_items_override_destination = True,
    
    # running parameters
    validation_options = "fail-pipeline",
    budget_milli_node_hours = 1000,
    
    # model parameters
    model_display_name = f"{SERIES}_{EXPERIMENT}",
    model_labels = {'series' : f'{SERIES}', 'experiment' : f'{EXPERIMENT}'},
    model_id = f"model_{SERIES}_{EXPERIMENT}",
    parent_model = parent,
    is_default_version = True,
    
    # session parameters: False means continue in local session, True waits and logs progress
    sync = False
)

wait until previous job is complete...

In [31]:
forecast.display_name, forecast.resource_name

('applied-forecasting_tide-python-v1',
 'projects/934903580331/locations/us-central1/models/model_applied-forecasting_tide-python-v1')

In [32]:
print(f'Review the model in the Vertex AI Model Registry:\nhttps://console.cloud.google.com/vertex-ai/locations/{REGION}/models/{forecast.name}?project={PROJECT_ID}')


Review the model in the Vertex AI Model Registry:
https://console.cloud.google.com/vertex-ai/locations/us-central1/models/model_applied-forecasting_tide-python-v1?project=hybrid-vertex


# Forecasting

## Retrieve Test Data

Start with a short sample. Notice the `TIME_COLUMN` (starttime) has repeating rows as the dates progress. The build up to the size of the context window. Also notice the column `predicted_on_####` which has date of the context.

In [33]:
query = f"""
SELECT *
FROM `{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_eval`
ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
LIMIT 20
"""
bq.query(query = query).to_dataframe()

,avg_tripduration,capacity,num_trips,pct_subscriber,predicted_num_trips,predicted_on_starttime,ratio_gender,start_station_name,starttime
0,1808.631970,36,269,0.423792,{'value': 144.6659698486328},2016-09-17,0.469945,Central Park North & Adam Clayton Powell Blvd,2016-09-17
1,2027.411765,36,272,0.367647,{'value': 157.0643768310547},2016-09-17,0.346535,Central Park North & Adam Clayton Powell Blvd,2016-09-18
2,2027.411765,36,272,0.367647,{'value': 139.8966522216797},2016-09-18,0.346535,Central Park North & Adam Clayton Powell Blvd,2016-09-18
3,1203.820513,36,39,0.743590,{'value': 148.56358337402344},2016-09-17,1.600000,Central Park North & Adam Clayton Powell Blvd,2016-09-19
4,1203.820513,36,39,0.743590,{'value': 143.88453674316406},2016-09-18,1.600000,Central Park North & Adam Clayton Powell Blvd,2016-09-19
5,1203.820513,36,39,0.743590,{'value': 125.07069396972656},2016-09-19,1.600000,Central Park North & Adam Clayton Powell Blvd,2016-09-19
6,1750.691667,36,120,0.625000,{'value': 126.62114715576172},2016-09-20,0.578947,Central Park North & Adam Clayton Powell Blvd,2016-09-20
7,1750.691667,36,120,0.625000,{'value': 142.68392944335938},2016-09-19,0.578947,Central Park North & Adam Clayton Powell Blvd,2016-09-20
8,1750.691667,36,120,0.625000,{'value': 145.33348083496094},2016-09-17,0.578947,Central Park North & Adam Clayton Powell Blvd,2016-09-20
9,1750.691667,36,120,0.625000,{'value': 153.77627563476562},2016-09-18,0.578947,Central Park North & Adam Clayton Powell Blvd,2016-09-20


Calculate the time between `TIME_COLUMN` and `prediction_on{TIME_COLUMN}` and keep the rows that are 0 = same day

In [34]:
query = f"""
SELECT
    DATE({TIME_COLUMN}) as {TIME_COLUMN},
    DATE(predicted_on_{TIME_COLUMN}) as predicted_on_{TIME_COLUMN},
    CAST({TARGET_COLUMN} as INT64) AS {TARGET_COLUMN},
    {SERIES_COLUMN},
    predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
FROM `{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_eval`
WHERE {TIME_COLUMN} = predicted_on_{TIME_COLUMN}
ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
"""
test_predictions = bq.query(query = query).to_dataframe()
test_predictions

,starttime,predicted_on_starttime,num_trips,start_station_name,predicted_num_trips
0,2016-09-17,2016-09-17,269,Central Park North & Adam Clayton Powell Blvd,144.665970
1,2016-09-18,2016-09-18,272,Central Park North & Adam Clayton Powell Blvd,139.896652
2,2016-09-19,2016-09-19,39,Central Park North & Adam Clayton Powell Blvd,125.070694
3,2016-09-20,2016-09-20,120,Central Park North & Adam Clayton Powell Blvd,126.621147
4,2016-09-21,2016-09-21,164,Central Park North & Adam Clayton Powell Blvd,130.471420
...,...,...,...,...,...
154,2016-09-26,2016-09-26,102,W 82 St & Central Park West,90.368507
155,2016-09-27,2016-09-27,105,W 82 St & Central Park West,85.905434
156,2016-09-28,2016-09-28,72,W 82 St & Central Park West,84.020912
157,2016-09-29,2016-09-29,143,W 82 St & Central Park West,79.659241


## Review Custom Metrics with SQL

Some common metrics for evaluating forecasting effectiveness are 
- MAPE, or Mean Absolute Percentage Error
    - $\textrm{MAPE} = \frac{1}{n}\sum{\frac{\mid(actual - forecast)\mid}{actual}}$
- MAE, or Mean Absolute Error
     - $\textrm{MAE} = \frac{1}{n}\sum{\mid(actual - forecast)\mid}$
- MAE divided by average demand so it yields a % like MAPE
    - $\textrm{pMAE} = \frac{\sum{\mid(actual - forecast)\mid}}{\sum{actual}}$
- MSE, or Mean Squared Error
    - $\textrm{MSE} = \frac{1}{n}\sum{(actual-forecast)^2}$
- RMSE, or Root Mean Squared Error
    - $\textrm{RMSE} = \sqrt{\frac{1}{n}\sum{(actual-forecast)^2}}$
- RMSE divided by average demand so it yeilds a % like MAPE
    - $\textrm{pRMSE} = \frac{\sqrt{\frac{1}{n}\sum{(actual-forecast)^2}}}{\frac{1}{n}\sum{actual}}$

It can be helpful to explicity calculate these to make comparison between datasets and models fair.  This section demonstration these calculation with SQL.

>```sql
>(actual_value - forecast_value) as diff
>
>
>AVG(SAFE_DIVIDE(ABS(diff), actual_value)) as MAPE,
>AVG(ABS(diff)) as MAE,
>SAFE_DIVIDE(SUM(ABS(diff)), SUM(actual_value)) as pMAE,
>AVG(POW(diff, 2)) as MSE,
>SQRT(AVG(POW(diff, 2))) as RMSE,
>SAFE_DIVIDE(SQRT(AVG(POW(diff, 2))), AVG(actual_value)) as pRMSE
>```

In [35]:
query = f"""
WITH
    FORECASTS AS (
        SELECT
            DATE({TIME_COLUMN}) as {TIME_COLUMN},
            DATE(predicted_on_{TIME_COLUMN}) as predicted_on_{TIME_COLUMN},
            CAST({TARGET_COLUMN} as INT64) AS {TARGET_COLUMN},
            {SERIES_COLUMN},
            predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
        FROM `{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_eval`
        WHERE {TIME_COLUMN} = predicted_on_{TIME_COLUMN}
    ),
    DIFFS AS (
        SELECT 
            {SERIES_COLUMN},
            {TIME_COLUMN},
            'forecast' as time_series_type,
            predicted_{TARGET_COLUMN} as forecast_value,
            {TARGET_COLUMN} as actual_value,
            ({TARGET_COLUMN} - predicted_{TARGET_COLUMN}) as diff
        FROM FORECASTS    
    )
SELECT
    start_station_name,
    time_series_type, 
    AVG(SAFE_DIVIDE(ABS(diff), actual_value)) as MAPE,
    AVG(ABS(diff)) as MAE,
    SAFE_DIVIDE(SUM(ABS(diff)), SUM(actual_value)) as pMAE,
    AVG(POW(diff, 2)) as MSE,
    SQRT(AVG(POW(diff, 2))) as RMSE,
    SAFE_DIVIDE(SQRT(AVG(POW(diff, 2))), AVG(actual_value)) as pRMSE
FROM DIFFS
GROUP BY
    {SERIES_COLUMN},
    time_series_type
ORDER BY
    {SERIES_COLUMN},
    time_series_type    
"""
# print(query)

customMetrics = bq.query(query = query).to_dataframe()
customMetrics

,start_station_name,time_series_type,MAPE,MAE,pMAE,MSE,RMSE,pRMSE
0,Central Park North & Adam Clayton Powell Blvd,forecast,0.706291,64.983219,0.418668,6811.801062,82.533636,0.531740
1,Central Park S & 6 Ave,forecast,0.647523,92.716341,0.284095,16110.797139,126.928315,0.388925
2,Central Park W & W 96 St,forecast,0.573024,27.510859,0.274912,1281.668268,35.800395,0.357748
3,Central Park West & W 100 St,forecast,0.819344,14.664604,0.369918,320.601024,17.905335,0.451666
4,Central Park West & W 102 St,forecast,0.535050,13.297730,0.262579,297.477682,17.247541,0.340572
5,Central Park West & W 68 St,forecast,0.526754,47.533413,0.313752,3390.058521,58.224209,0.384318
6,Central Park West & W 72 St,forecast,0.651029,50.913189,0.287770,4721.858730,68.715782,0.388394
7,Central Park West & W 76 St,forecast,0.384649,26.804930,0.244794,1278.117662,35.750771,0.326491
8,Central Park West & W 85 St,forecast,1.215660,53.403264,0.424799,5018.465491,70.841129,0.563509
9,Grand Army Plaza & Central Park S,forecast,0.519924,45.682368,0.210071,5255.164933,72.492516,0.333358


### Overall Metrics:

In [36]:
query = f"""
WITH
    FORECASTS AS (
        SELECT
            DATE({TIME_COLUMN}) as {TIME_COLUMN},
            DATE(predicted_on_{TIME_COLUMN}) as predicted_on_{TIME_COLUMN},
            CAST({TARGET_COLUMN} as INT64) AS {TARGET_COLUMN},
            {SERIES_COLUMN},
            predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
        FROM `{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_eval`
        WHERE {TIME_COLUMN} = predicted_on_{TIME_COLUMN}
    ),
    DIFFS AS (
        SELECT 
            {SERIES_COLUMN},
            {TIME_COLUMN},
            'forecast' as time_series_type,
            predicted_{TARGET_COLUMN} as forecast_value,
            {TARGET_COLUMN} as actual_value,
            ({TARGET_COLUMN} - predicted_{TARGET_COLUMN}) as diff
        FROM FORECASTS    
    )
SELECT
    #start_station_name,
    time_series_type, 
    AVG(SAFE_DIVIDE(ABS(diff), actual_value)) as MAPE,
    AVG(ABS(diff)) as MAE,
    SAFE_DIVIDE(SUM(ABS(diff)), SUM(actual_value)) as pMAE,
    AVG(POW(diff, 2)) as MSE,
    SQRT(AVG(POW(diff, 2))) as RMSE,
    SAFE_DIVIDE(SQRT(AVG(POW(diff, 2))), AVG(actual_value)) as pRMSE
FROM DIFFS
GROUP BY
    #{SERIES_COLUMN},
    time_series_type
ORDER BY
    #{SERIES_COLUMN},
    time_series_type  
"""
# print(query)

customMetricsOverall = bq.query(query = query).to_dataframe()
customMetricsOverall

,time_series_type,MAPE,MAE,pMAE,MSE,RMSE,pRMSE
0,forecast,0.58734,39.625146,0.295863,3812.736363,61.747359,0.461039


# Get Forecasted Values for Future Horizon

Use a batch prediction job with the resulting forecasting model to get predicted forecast for the future horizon.

The requirement for getting batch predictions from a Vertex AI forecasting model are covered [here](https://cloud.google.com/vertex-ai/docs/tabular-data/forecasting/get-predictions)

## Prepare Input Table in BigQuery

For Vetex AI forecasting batch prediction jobs we need a table (BigQuery or CSV) that contains a row per date in the forecast horizon and history for atleast the length of the context window. The following BigQuery query constructs this input table from the source data while also filling in missing dates in the context window with the last known observation.

In [38]:
context_window = 28

query_a = ""
query_b = ""
for v in COVARIATE_COLUMNS_KNOWN + COVARIATE_COLUMNS_UNKNOWN + COVARIATE_COLUMNS_ATTRIBUTES:
    query_a += f""",
            LAST_VALUE({v} IGNORE NULLS) OVER (PARTITION BY {SERIES_COLUMN} ORDER BY {TIME_COLUMN} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as {v}"""
    if v not in COVARIATE_COLUMNS_ATTRIBUTES:
        query_b += f""",
        CASE WHEN {TIME_COLUMN} > (SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`) THEN NULL ELSE {v} END AS {v}"""
    else:
        query_b += f""",
        {v}"""

query = f"""
CREATE OR REPLACE TABLE `{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_horizon_input` AS
WITH
    DATELIST AS (
        SELECT *
        FROM (SELECT DISTINCT {SERIES_COLUMN} FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`) A
        CROSS JOIN (SELECT * 
                    FROM UNNEST(GENERATE_DATE_ARRAY(
                                    DATE_SUB((SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`), INTERVAL {context_window-1} DAY),
                                    DATE_ADD((SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`), INTERVAL {FORECAST_HORIZON_LENGTH} DAY),
                                    INTERVAL 1 DAY
                                )
                            ) AS {TIME_COLUMN}
                    ) B
    ),
    ADDTARGET AS (
        SELECT *
        FROM DATELIST
        LEFT OUTER JOIN (SELECT * FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`)
        USING ({SERIES_COLUMN}, {TIME_COLUMN})
        ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
    ),
    LOCF AS (
        SELECT {SERIES_COLUMN}, {TIME_COLUMN},
        LAST_VALUE({TARGET_COLUMN} IGNORE NULLS) OVER (PARTITION BY {SERIES_COLUMN} ORDER BY {TIME_COLUMN} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as {TARGET_COLUMN}
        {query_a}
        FROM ADDTARGET
    )
SELECT {SERIES_COLUMN}, {TIME_COLUMN},
    CASE
        WHEN {TIME_COLUMN} > (SELECT MAX({TIME_COLUMN}) FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`) THEN NULL
        ELSE {TARGET_COLUMN}
    END AS {TARGET_COLUMN}
    {query_b}
FROM LOCF
ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
"""

# print(query)

job = bq.query(query = query)
job.result()

## Batch Prediction Job

> Request the batch prediction directly using the `batch_prediction()` method for the model - [SDK Reference](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_batch_predict)

In [39]:
batchjob = forecast.batch_predict(
    job_display_name = f'{SERIES}_{EXPERIMENT}_{TIMESTAMP}',
    bigquery_source = f"bq://{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_horizon_input",
    bigquery_destination_prefix = f"bq://{BQ_PROJECT}.{BQ_DATASET}",
    sync = False
)

#### Process Predicted Forecast

In [40]:
batchjob.output_info.bigquery_output_table

'predictions_2025_03_31T18_29_39_917Z_877'

In [41]:
query = f"""
    CREATE OR REPLACE TABLE `{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_horizon_output` AS
    SELECT {SERIES_COLUMN}, DATE({TIME_COLUMN}) as {TIME_COLUMN}, predicted_{TARGET_COLUMN}.value as predicted_{TARGET_COLUMN}
    FROM `{BQ_PROJECT}.{BQ_DATASET}.{batchjob.output_info.bigquery_output_table}`
"""
job = bq.query(query = query)
job.result()

In [42]:
query = f"""
    SELECT *
    FROM `{BQ_PROJECT}.{BQ_DATASET}.{EXPERIMENT}_horizon_output`
    ORDER BY {SERIES_COLUMN}, {TIME_COLUMN}
"""
predict = bq.query(query = query).to_dataframe()
predict.head()

,start_station_name,starttime,predicted_num_trips
0,Central Park North & Adam Clayton Powell Blvd,2016-10-01,140.611496
1,Central Park North & Adam Clayton Powell Blvd,2016-10-02,155.339798
2,Central Park North & Adam Clayton Powell Blvd,2016-10-03,119.524269
3,Central Park North & Adam Clayton Powell Blvd,2016-10-04,98.750130
4,Central Park North & Adam Clayton Powell Blvd,2016-10-05,105.143211


# Visualize The Time Series With Forecast

In [48]:
viz_limit = 12

In [50]:
# NA values in Pandas will not convert to JSON which Plotly uses:
rawSeries = rawSeries.fillna(np.nan).replace([np.nan], [None])

# create a figure:
fig = go.Figure()

# get a list of colors to use:
colors = px.colors.qualitative.Plotly

# list of columns to plot over time : target and covariates
variables = [TARGET_COLUMN] #+ COVARIATE_COLUMNS

# create dropdown/button to toggle series
buttons = []
b = 0 # default button index

# iterate through series:
series = rawSeries[SERIES_COLUMN].unique().tolist()[0:viz_limit]
for s in series:    
    # iterate trhough columns
    for y, v in enumerate(variables):
        fig.add_trace(
            go.Scatter(
                x = rawSeries[rawSeries[SERIES_COLUMN]==s][TIME_COLUMN],
                y = rawSeries[rawSeries[SERIES_COLUMN]==s][v],
                name = f'{v}',
                text = rawSeries[rawSeries[SERIES_COLUMN]==s][v],
                yaxis = f"y{y+1}",
                hoverinfo='name+x+text',
                line = {'width': 0.5},
                marker = {'size': 8},
                mode = 'lines+markers',
                showlegend = False,
                visible = (b==0) # make a series visible as default: this uses the first series
            )
        )
        if y == 0: # add the forecast
            # add the forecast fit
            fig.add_trace(
                go.Scatter(
                    x = test_predictions[test_predictions[SERIES_COLUMN]==s][TIME_COLUMN],
                    y = test_predictions[test_predictions[SERIES_COLUMN]==s][f'predicted_{TARGET_COLUMN}'],
                    name = f'Forecast: {v}',
                    text = test_predictions[test_predictions[SERIES_COLUMN]==s][f'predicted_{TARGET_COLUMN}'],
                    yaxis = f"y{y+1}",
                    hoverinfo='name+x+text',
                    line = {'width': 2, 'color': 'rgb(255,234,0)'},
                    mode = 'lines',
                    showlegend = False,
                    visible = (b==0) # make a series visible as default: this uses the first series
                )
            )
            fig.add_trace(
                go.Scatter(
                    x = predict[predict[SERIES_COLUMN]==s][TIME_COLUMN],
                    y = predict[predict[SERIES_COLUMN]==s][f'predicted_{TARGET_COLUMN}'],
                    name = f'Forecast: {v}',
                    text = predict[predict[SERIES_COLUMN]==s][f'predicted_{TARGET_COLUMN}'],
                    yaxis = f"y{y+1}",
                    hoverinfo='name+x+text',
                    line = {'width': 2, 'color': 'rgb(255,234,0)'},
                    mode = 'lines',
                    showlegend = False,
                    visible = (b==0) # make a series visible as default: this uses the first series
                )
            )
    
    # which button to show:
    ff = 2 # count of forecast related traces add to each series
    which_buttons = [False] * len(series) * (len(variables) + ff)
    which_buttons[b * (len(variables) +ff):(b+1)*(len(variables) + ff)] = [True] * (len(variables) + ff)

    # create button for series:
    button = dict(
        label = s,
        method = 'update',
        args = [{'visible': which_buttons}]
    )
    buttons.append(button)
    b += 1

# add split regions: training
fig.add_shape(
    fillcolor = 'rgba(0, 255, 0, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['start_date'][0],
    x1 = keyDates['val_start'][0],
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['val_start'][0] - (keyDates['test_start'][0]-keyDates['val_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Training',
    yanchor = 'bottom'
)

# add split regions: validation
fig.add_shape(
    fillcolor = 'rgba(255, 255, 0, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['val_start'][0],
    x1 = keyDates['test_start'][0],
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['val_start'][0] + (keyDates['test_start'][0]-keyDates['val_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Validation',
    yanchor = 'bottom'
)

# add split regions: test
fig.add_shape(
    fillcolor = 'rgba(0, 0, 255, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['test_start'][0],
    x1 = keyDates['end_date'][0],
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['test_start'][0] + (keyDates['end_date'][0]-keyDates['test_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Test',
    yanchor = 'bottom'
)

# add split regions: horizon
fig.add_shape(
    fillcolor = 'rgba(255, 255, 255, 0.2)',
    line = {'width': 0},
    type = 'rect',
    x0 = keyDates['end_date'][0],
    x1 = keyDates['end_date'][0]+timedelta(days = FORECAST_HORIZON_LENGTH),
    xref = 'x',
    y0 = 0,
    y1 = 1,
    yref = 'paper'
)
fig.add_annotation(
    x = keyDates['end_date'][0] + (keyDates['end_date'][0]-keyDates['test_start'][0])/2,
    y = 0,
    ax = 0, ay = 0,
    yref = 'y1',
    xref = 'x',
    text = 'Horizon',
    yanchor = 'bottom'
)

# configure axes layout:
layout = dict(
    xaxis =  dict(
        range = [keyDates['end_date'][0] - 2*(keyDates['end_date'][0] - keyDates['val_start'][0]), keyDates['end_date'][0]+timedelta(days = FORECAST_HORIZON_LENGTH)],
        rangeslider = dict(
            autorange = True,
            range = [keyDates['start_date'][0], keyDates['end_date'][0]+timedelta(days = FORECAST_HORIZON_LENGTH)]
        ),
        type = 'date'
    )
)
for v, variable in enumerate(variables):
    layout[f'yaxis{v+1}'] = dict(
        anchor = 'x',
        domain = [v*(1/len(variables)), (v+1)*(1/len(variables))],
        autorange = True,
        mirror = True,
        autoshift = True,
        title = dict(text = variable, standoff = 10 + 20 * (v % 2), font = dict(color = colors[v])),
        tickfont = dict(color = colors[v]),
        tickmode = 'auto',
        linecolor = colors[v],
        linewidth = 4,
        showline = True,
        side = 'right',
        type = 'linear',
        zeroline = False
    )

# final update of display before rendering
fig.update_layout(
    layout,
    title = 'Time Series Plots:',
    dragmode="zoom",
    hovermode="x",
    legend=dict(traceorder="reversed"),
    height=600,
    template="plotly_white",
    margin=dict(
        t=100,
        b=100
    ),
    updatemenus = [
        dict(
            buttons = buttons,
            type = 'dropdown',
            direction = 'down',
            x = 1,
            y = 1.2,
            showactive = True
        )
    ]
)

# render the interactive plot:
fig.show()

In [ ]:
# print('A Snapshot of the interactive plot:')
# fig.show('png')

**Finished**